# Blood Lab Data Cleaning

In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import re
import requests
from bs4 import BeautifulSoup

In [7]:
!pip install pyreadstat
import pyreadstat #since the data files are .xpt files, this library is needed to import the table
from nhanes_utils import to_snake_case, get_common_nan_ids, standardize_id_column, drop_rows_with_common_nan_ids

Access is denied.


ModuleNotFoundError: No module named 'pyreadstat'

### Alpha-1-acid glycoprotein (AGP)

Alpha-1-acid glycoprotein (AGP), also known as orosomucoid (ORM), is an acute-phase serum protein present in humans and many animal species. It is produced in response to inflammation, although its precise biological role remains under investigation and somewhat ambiguous [2]. According to Ceciliani et al. (2019), AGP may play a role in immunometabolism, a function potentially relevant to understanding the obesity epidemic in the U.S.

In the NHANES dataset, AGP levels were measured in children aged 3–5 years and females aged 12–49 years. This data offers an opportunity to explore potential correlations between AGP serum concentrations and obesity prevalence among the female participants in the study.

In [ ]:
file_path = '2017-2020/blood/1.P_SSAGP.xpt'

df_b1, meta = pyreadstat.read_xport(file_path)
df_b1 = standardize_id_column(df_b1)

In [ ]:
df_b1.head(10)

In [ ]:
df_b1.shape

In [ ]:
df_b1 = df_b1.rename(columns={
    'SSAGP': 'alpha_1_agp_g_l'
})

In [ ]:
df_b1.head()

In [ ]:
df_b1 = df_b1.drop('WTSSAGPP', axis=1)

In [ ]:
df_b1 = df_b1.dropna(subset = ['alpha_1_agp_g_l'])

In [ ]:
df_b1.head(5)

### Lipid Panel

Lipids are essential molecules that support a range of physiological functions, including hormone production and cellular structure. However, excessive lipid levels—particularly certain types—are associated with increased risk of cardiovascular disease.

To assess lipid status, a fasting lipid panel is commonly used. This test typically includes measurements of:
- LDL (low-density lipoprotein, or “bad” cholesterol),
- HDL (high-density lipoprotein, or “good” cholesterol),
- Total cholesterol, and
- Triglycerides

The NHANES dataset includes all of these values, enabling analysis of lipid profiles across a large representative population. This section focuses on cleaning and preparing these variables for analysis.

In [ ]:
file_path = '2017-2020/blood/2.P_HDL.xpt'

df_b2, meta = pyreadstat.read_xport(file_path)
df_b2 = standardize_id_column(df_b2)

In [ ]:
df_b2.head()

In [ ]:
df_b2 = df_b2.rename(columns ={
    'LBDHDD':'direct_hdl_mg_dl',
    'LBDHDDSI':'direct_hdl_mmol_l'
})

In [ ]:
df_b2.isnull().sum()

In [ ]:
common_nan = get_common_nan_ids(df_b2, 'direct_hdl_mg_dl', 'direct_hdl_mmol_l', id_col='participant_id')

In [ ]:
df_b2 = drop_rows_with_common_nan_ids(df_b2, 'direct_hdl_mg_dl', 'direct_hdl_mmol_l', id_col='participant_id')

In [ ]:
df_b2.to_csv('hdl_cleaned.csv')

In [ ]:
file_path = '2017-2020/blood/3.P_TRIGLY.xpt'

df_b3, meta = pyreadstat.read_xport(file_path)
df_b3 = standardize_id_column(df_b3)

In [ ]:
df_b3.columns.to_list()

In [ ]:
df_b3 = df_b3.rename(columns={
    'LBXTR':'triglyceride_mg_dl',
    'LBDTRSI':'triglyceride_mmol_l',
    'LBDLDL':'ldl_friedewald_mg_dl',
    'LBDLDLSI':'ldl_friedwalkd_mmol_l',
    'LBDLDLM': 'ldl_martin_hopkins_mg_dl',
    'LBDLDMSI': 'ldl_martin_hopkins_mmol_l',
    'LBDLDLN':'ldl_nih_mg_dl',
    'LBDLDNSI':'ldl_nih_mmol_l'
})

In [ ]:
df_b3 = df_b3.drop('WTSAFPRP', axis=1)

In [ ]:
df_b3.isnull().sum()

In [ ]:
value_cols = [col for col in df_b3.columns if col != 'participant_id']
rows_all_nan = df_b3[value_cols].isna().all(axis=1)
print(f"Number of rows missing all cholesterol values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b3_cleaned = df_b3[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

In [ ]:
df_b3.to_csv('tri_ldl_cleaned.csv')

In [ ]:
file_path = '2017-2020/blood/4.P_TCHOL.xpt'

df_b4, meta = pyreadstat.read_xport(file_path)
df_b4 = standardize_id_column(df_b4)

In [ ]:
df_b4.head()

In [ ]:
df_b4 = df_b4.rename(columns={
    'LBXTC': 'total_cholesterol_mg_dl',
    'LBDTCSI':'total_cholesterol_mmol_l'
})

In [ ]:
df_b4.isnull().sum()

In [ ]:
common_nan = get_common_nan_ids(df_b4, 'total_cholesterol_mg_dl', 'total_cholesterol_mmol_l', id_col='participant_id')

In [ ]:
df_b4 = drop_rows_with_common_nan_ids(df_b4, 'total_cholesterol_mg_dl', 'total_cholesterol_mmol_l', id_col='participant_id')

In [ ]:
df_b4.to_csv('total_chol_cleaned.csv')

### Chromium and Cobalt (Blood)

NHANES data on chromium and cobalt levels were collected on patients aged 40-150 years old. 

In [ ]:
file_path = '2017-2020/blood/5.P_CRCO.xpt'

df_b5, meta = pyreadstat.read_xport(file_path)
df_b5 = standardize_id_column(df_b5)

In [ ]:
df_b5.columns.to_list()

In [ ]:
df_b5 = df_b5.rename(columns={
    'LBXBCR':'chromium_blood_ug_l', 
    'LBDBCRSI': 'chromium_blood_nmol_l',
    'LBDBCRLC':'chromium_blood_comment', 
    'LBXBCO':'cobalt_blood_ug_l', 
    'LBDBCOSI':'cobalt_blood_nmol_l', 
    'LBDBCOLC' :'cobalt_blood_comment'
})

In [ ]:
df_b5.isnull().sum()

In [ ]:
common_nan = get_common_nan_ids(df_b5, 'chromium_blood_ug_l', 'cobalt_blood_ug_l', id_col='participant_id')

In [ ]:
df_b5 = drop_rows_with_common_nan_ids(df_b5, 'chromium_blood_ug_l', 'cobalt_blood_ug_l', id_col='participant_id')

### Complete Blood Count with Differential

CBC with diff is the most common blood work that is ordered for a baseline lab. CBC can be useful to assess the patients for acute inflammation in the body and anemia. 

There are many values that are extracted and assessed through the CBC panel. To more efficiently extract the information, the decision was made to utilize webscraping rather than individually typing out each lab value. The units for each of the columns is included in the README file for this project.

In [ ]:
file_path = '2017-2020/blood/6.P_CBC.xpt'

df_b6, meta = pyreadstat.read_xport(file_path)
df_b6 = standardize_id_column(df_b6)

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_CBC.htm"

df_info_raw = pd.read_html(url)[0]

# Use the first row as the header
df_info_raw.columns = df_info_raw.iloc[0]
df_info = df_info_raw.drop(index=0).reset_index(drop=True)

In [ ]:
df_info.head()

In [ ]:
df_info.columns.to_list()

In [ ]:
rename_dict = {
    row["Variable  Name"]: to_snake_case(row["Analyte  Description"])
    for _, row in df_info.iterrows()
    if row["Variable  Name"] in df_b6.columns
}

In [ ]:
df_b6 = df_b6.rename(columns=rename_dict)

In [ ]:
df_b6.head()

In [ ]:
df_b6.columns.to_list()

#Some of the names were renamed based on what was available on the first table in the URL. For the other ones that were not, they were manually renamed

In [ ]:
df_b6 = df_b6.rename(columns={
 'LBDLYMNO': 'lymphocyte_number',
 'LBDMONO': 'monocyte_number',
 'LBDNENO':'segmented_neutrophils_number',
 'LBDEONO':'eosinophils_number',
 'LBDBANO':'basophils_number',
 'LBXHCT':'hematocrit_percent',
 'LBXMC':'mean_cell_hgb_concentration',
 'LBXMCHSI':'mean_cell_hemoglobin',
 'LBXNRBC':'nucelated_red_blood_cells'
})

In [ ]:
df_b6.columns

In [ ]:
df_b6.isnull().sum()

In [ ]:
value_cols = [col for col in df_b6.columns if col != 'participant_id']
rows_all_nan = df_b6[value_cols].isna().all(axis=1)
print(f"Number of rows missing all CBC values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b6_cleaned = df_b6[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

In [ ]:
df_b6_cleaned.to_csv('blood_cbc.csv')

### Cotinine

Cotinine is a metabolite that is produced when nicotine is processed. Its long half-life makes it a good marker for assessing tobacco exposure or usage. 

In [ ]:
file_path = '2017-2020/blood/7.P_COT.xpt'

df_b7, meta = pyreadstat.read_xport(file_path)
df_b7 = standardize_id_column(df_b7)

In [ ]:
df_b7.columns.to_list()

In [ ]:
df_b7 = df_b7.rename(columns={
    'LBXCOT':'serum_cotinine_ng_ml',
    'LBDCOTLC':'serum_cotinine_comment',
    'LBXHCOT':'serum_hydroxycotinine_ng_ml',
    'LBDHCOLC':'serum_hydroxycotinine_comment'
})

In [ ]:
df_b7.isnull().sum()

In [ ]:
common_nan = get_common_nan_ids(df_b7, 'serum_cotinine_ng_ml', 'serum_hydroxycotinine_ng_ml', id_col='participant_id')

In [ ]:
df_b7 = drop_rows_with_common_nan_ids(df_b7, 'serum_cotinine_ng_ml', 'serum_hydroxycotinine_ng_ml', id_col='participant_id')

### Cytomegalovirus

Cytomegalovirus (CMV) is a double-stranded DNA virus that causes flu-like symptoms in immunocompetant population but can cause organ damage in immunocompromised (i.e. HIV/AIDS) population. CMV virus is transmitted via bodily fluids including sexual contact [12]. 

Avidity tests for whether the CMV infection was recent or in the past. Low avidity shows recent infection and high avidity shows past infeciton.

In [ ]:
file_path = '2017-2020/blood/8.P_CMV.xpt'

df_b8, meta = pyreadstat.read_xport(file_path)
df_b8 = standardize_id_column(df_b8)

In [ ]:
df_b8.columns.to_list()

In [ ]:
df_b8 = df_b8.rename(columns={
    'LBXIGG':'cmv_igg',
    'LBXIGM':'cmv_igm', 
    'LBXIGGA':'cmv_igg_avidity'
})

In [ ]:
df_b8.isnull().sum()

#there are missing avidity value which would indicate that the person was never infected with CMV. The null values for the IgG and IgM would indicate missing data so rows without these two values will be dropped

In [ ]:
common_nan = get_common_nan_ids(df_b8, 'cmv_igg', 'cmv_igm', id_col='participant_id')

In [ ]:
df_b8 = drop_rows_with_common_nan_ids(df_b8, 'cmv_igg', 'cmv_igm', id_col='participant_id')

### Ethylene Oxide

Ethylene Oxide (EtO) is a colorless gas that is used to produce various materials as well as sterilize medical equipments. Exposure to EtO most often is due to aerosolization. EtO is a well-known carcinogen and long term exposure to this substance could lead to blood cancers such as non-Hodgkin lymphoma, myeloma and lymphocytic leukemia [13].

The unit for EtO measurement in the blood is picomoles per gram of hemoglobin (pmol/g Hb).

In [ ]:
file_path = '2017-2020/blood/9.P_ETHOX.xpt'

df_b9, meta = pyreadstat.read_xport(file_path)
df_b9 = standardize_id_column(df_b9)

In [ ]:
df_b9.columns.to_list()

In [ ]:
df_b9 = df_b9.drop('WTSAPRP',axis=1)

In [ ]:
df_b9 = df_b9.rename(columns={
    'LBXEOA':'eto_pmol_g_hb',
    'LBDEOALC':'eto_comment'
})

In [ ]:
df_b9.isnull().sum()

In [ ]:
df_b9 = df_b9.dropna(subset=['eto_pmol_g_hb'])

In [ ]:
df_b9.head()

### Ferritin and iron panel

Ferritin and iron panel are used to assess someone's iron status. Low values in the iron panel and ferritin along with clinical symptoms are corroborated to diagnose iron deficiency anemia. 

In [ ]:
file_path = '2017-2020/blood/10.P_FERTIN.xpt'

df_b10, meta = pyreadstat.read_xport(file_path)
df_b10 = standardize_id_column(df_b10)

In [ ]:
df_b10.head()

In [ ]:
df_b10 = df_b10.rename(columns={
    'LBXFER':'ferritin_ng_ml',
    'LBDFERSI':'ferritin_ug_l'
})

In [ ]:
df_b10.isnull().sum()

In [ ]:
common_nan = get_common_nan_ids(df_b10, 'ferritin_ng_ml', 'ferritin_ug_l', id_col='participant_id')

In [ ]:
df_b10 = drop_rows_with_common_nan_ids(df_b10, 'ferritin_ng_ml', 'ferritin_ug_l', id_col='participant_id')

In [ ]:
file_path = '2017-2020/blood/11.P_FETIB.xpt'

df_b11, meta = pyreadstat.read_xport(file_path)
df_b11 = standardize_id_column(df_b11)

In [ ]:
df_b11.columns.to_list()

In [ ]:
df_b11 = df_b11.rename(columns={
 'LBXIRN':'iron_frozen_ug_dl',
 'LBDIRNSI':'iron_frozen_umol_l',
 'LBXUIB':'uibc_ug_dl',
 'LBDUIBLC':'uibc_comment',
 'LBDUIBSI':'uibc_umol_l',
 'LBDTIB':'tibc_ug_dl',
 'LBDTIBSI':'tibc_umol_l',
 'LBDPCT':'transferrin_saturation'
})

In [ ]:
df_b11.isnull().sum()

In [ ]:
value_cols = [col for col in df_b11.columns if col != 'participant_id']
rows_all_nan = df_b11[value_cols].isna().all(axis=1)
print(f"Number of rows missing all iron panel values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b11_cleaned = df_b11[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

In [ ]:
file_path = '2017-2020/blood/23.P_TFR.xpt'

df_b23, meta = pyreadstat.read_xport(file_path)
df_b23 = standardize_id_column(df_b23)

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_TFR.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [ ]:
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df_b23.columns
}

df_b23 = df_b23.rename(columns=filtered_rename_dict)

In [ ]:
df_b23.head()

In [ ]:
df_b23.isnull().sum()

In [ ]:
df_b23 = df_b23.dropna()

In [ ]:
df_b23.head()

### Folate


In [ ]:
file_path = '2017-2020/blood/12.P_FOLATE.xpt'

df_b12, meta = pyreadstat.read_xport(file_path)
df_b12 = standardize_id_column(df_b12)

In [ ]:
df_b12.columns

In [ ]:
df_b12 = df_b12.drop('WTFOLPRP', axis=1)

In [ ]:
df_b12 = df_b12.rename(columns={
    'LBDRFO':'rbc_folate_ng_ml',
    'LBDRFOSI':'rbc_folate_nmol_l'
})

In [ ]:
df_b12.isnull().sum()

In [ ]:
common_nan = get_common_nan_ids(df_b12, 'rbc_folate_ng_ml', 'rbc_folate_nmol_l', id_col='participant_id')

In [ ]:
df_b12 = drop_rows_with_common_nan_ids(df_b12, 'rbc_folate_ng_ml', 'rbc_folate_nmol_l', id_col='participant_id')

In [ ]:
file_path = '2017-2020/blood/13.P_FOLFMS.xpt'

df_b13, meta = pyreadstat.read_xport(file_path)
df_b13 = standardize_id_column(df_b13)

#There are a lot of technical names for these values so webscraping will be done instead of manual renaming of the columns

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_FOLFMS.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# Pattern: match things like "LBDRFOSI - RBC folate (nmol/L)"
pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [ ]:
# Only keep entries in rename_dict where the variable name is in df_b13
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df_b13.columns
}

# Rename columns in df_b13
df_b13 = df_b13.rename(columns=filtered_rename_dict)

In [ ]:
df_b13.head()

In [ ]:
df_b13.columns.to_list()

In [ ]:
df_b13 = df_b13.drop('folate_folate_form_weight_pre_pandemic',axis=1)

In [ ]:
df_b13.isnull().sum()

In [ ]:
value_cols = [col for col in df_b13.columns if col != 'participant_id']
rows_all_nan = df_b13[value_cols].isna().all(axis=1)
print(f"Number of rows missing all folate values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b13_cleaned = df_b13[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

### Glycohemoglobin (%)

In [ ]:
file_path = '2017-2020/blood/14.P_GHB.xpt'

df_b14, meta = pyreadstat.read_xport(file_path)
df_b14 = standardize_id_column(df_b14)

In [ ]:
df_b14.head()

In [ ]:
df_b14 = df_b14.rename(columns = {'LBXGH':'glycohemoglobin_percent'})

In [ ]:
df_b14.isnull().sum()

In [ ]:
df_b14 = df_b14.dropna(subset=['glycohemoglobin_percent'])

### High-Sensitivity C-Reactive Protein

In [ ]:
file_path = '2017-2020/blood/15.P_HSCRP.xpt'

df_b15, meta = pyreadstat.read_xport(file_path)
df_b15 = standardize_id_column(df_b15)

In [ ]:
df_b15.head()

In [ ]:
df_b15 = df_b15.rename(columns = {
    'LBXHSCRP':'hs_crp_mg_l',
    'LBDHRPLC':'hs_crp_cmt'
})

In [ ]:
df_b15.isnull().sum()

In [ ]:
df_b15 = df_b15.dropna(subset=['hs_crp_mg_l'])

### Inorganic metyl and ethyl mercury

In [ ]:
file_path = '2017-2020/blood/16.P_IHGEM.xpt'

df_b16, meta = pyreadstat.read_xport(file_path)
df_b16 = standardize_id_column(df_b16)

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_IHGEM.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [ ]:
# Only keep entries in rename_dict where the variable name is in df_b13
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df_b16.columns
}

# Rename columns in df_b13
df_b16 = df_b16.rename(columns=filtered_rename_dict)

In [ ]:
df_b16.head()

In [ ]:
df_b16.isnull().sum()

In [ ]:
value_cols = [col for col in df_b16.columns if col != 'participant_id']
rows_all_nan = df_b16[value_cols].isna().all(axis=1)
print(f"Number of rows missing all inorganic mercury values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b16_cleaned = df_b16[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

### Insulin

In [ ]:
file_path = '2017-2020/blood/17.P_INS.xpt'

df_b17, meta = pyreadstat.read_xport(file_path)
df_b17 = standardize_id_column(df_b17)

# this file may be corrupted?

In [ ]:
with open(file_path, "rb") as f:
    content = f.read()
    if b'\xb5' in content:
        print("Contains micro symbol (µ)")

In [ ]:
# Step 1: Read the raw .xpt file as bytes
with open(file_path, "rb") as f:
    content = f.read()

# Step 2: Replace µ (byte 0xB5) with 'u' or another safe character
cleaned = content.replace(b'\xb5', b'u')  # or b'mu' if you prefer

# Step 3: Save it to a new temporary file
with open(file_path, "wb") as f:
    f.write(cleaned)

# Step 4: Now read it with pyreadstat
df_b17, meta = pyreadstat.read_xport(file_path)
df_b17 = standardize_id_column(df_b17)

In [ ]:
df_b17.head()

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_INS.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [ ]:
# Only keep entries in rename_dict where the variable name is in df_b13
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df_b17.columns
}

# Rename columns in df_b13
df_b17 = df_b17.rename(columns=filtered_rename_dict)

In [ ]:
df_b17.head()

In [ ]:
df_b17.isnull().sum()

In [ ]:
df_b17 = df_b17.drop(['fasting_subsample_weight'],axis=1)

In [ ]:
value_cols = [col for col in df_b17.columns if col != 'participant_id']
rows_all_nan = df_b17[value_cols].isna().all(axis=1)
print(f"Number of rows missing all insulin values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b17_cleaned = df_b17[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

### Lead, Cadmium, Total Mercury, Selenium, & Manganese

In [ ]:
file_path = '2017-2020/blood/18.P_PBCD.xpt'

df_b18, meta = pyreadstat.read_xport(file_path)
df_b18 = standardize_id_column(df_b18)

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_PBCD.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [ ]:
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df_b18.columns
}

df_b18 = df_b18.rename(columns=filtered_rename_dict)

In [ ]:
df_b18.head()

In [ ]:
df_b18.isnull().sum()

In [ ]:
value_cols = [col for col in df_b18.columns if col != 'participant_id']
rows_all_nan = df_b18[value_cols].isna().all(axis=1)
print(f"Number of rows missing metal values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b18_cleaned = df_b18[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

### Perfluoroalkyl and Polyfluoroalkyl Substances

In [ ]:
file_path = '2017-2020/blood/19.P_PFAS.xpt'

df_b19, meta = pyreadstat.read_xport(file_path)
df_b19 = standardize_id_column(df_b19)

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_PFAS.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [ ]:
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df_b19.columns
}

# Rename columns in df_b13
df_b19 = df_b19.rename(columns=filtered_rename_dict)

In [ ]:
df_b19.head()

In [ ]:
df_b19 = df_b19.drop(['subsample_ba_weights_pre_pandemic'],axis=1)

In [ ]:
df_b19.isnull().sum()

In [ ]:
value_cols = [col for col in df_b19.columns if col != 'participant_id']
rows_all_nan = df_b19[value_cols].isna().all(axis=1)
print(f"Number of rows missing PFAS values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b19_cleaned = df_b19[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

### Fasting Glucose

In [ ]:
file_path = '2017-2020/blood/20.P_GLU.xpt'

df_b20, meta = pyreadstat.read_xport(file_path)
df_b20 = standardize_id_column(df_b20)

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_GLU.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [ ]:
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df_b20.columns
}

df_b20 = df_b20.rename(columns=filtered_rename_dict)

In [ ]:
df_b20.head()

In [ ]:
df_b20 = df_b20.drop(['fasting_subsample_weight'],axis=1)

In [ ]:
df_b20.isnull().sum()

In [ ]:
value_cols = [col for col in df_b20.columns if col != 'participant_id']
rows_all_nan = df_b20[value_cols].isna().all(axis=1)
print(f"Number of rows missing fasting glucose values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b20_cleaned = df_b20[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

### Sex Steroid Hormone Panel

In [ ]:
file_path = '2017-2020/blood/21.P_TST.xpt'

df_b21, meta = pyreadstat.read_xport(file_path)
df_b21 = standardize_id_column(df_b21)

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_TST.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [ ]:
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df_b21.columns
}

df_b21 = df_b21.rename(columns=filtered_rename_dict)

In [ ]:
df_b21.head()

In [ ]:
df_b21 = df_b21.drop(['tst_subsample_weights_pre_pandemic'],axis=1)

In [ ]:
df_b21.isnull().sum()

In [ ]:
value_cols = [col for col in df_b21.columns if col != 'participant_id']
rows_all_nan = df_b21[value_cols].isna().all(axis=1)
print(f"Number of rows missing sex hormone values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b21_cleaned = df_b21[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

### Complete Metabolic Panel

In [ ]:
file_path = '2017-2020/blood/22.P_BIOPRO.xpt'

df_b22, meta = pyreadstat.read_xport(file_path)
df_b22 = standardize_id_column(df_b22)

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BIOPRO.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [ ]:
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df_b22.columns
}

df_b22 = df_b22.rename(columns=filtered_rename_dict)

In [ ]:
df_b22.head()

In [ ]:
df_b22.isnull().sum()

In [ ]:
value_cols = [col for col in df_b22.columns if col != 'participant_id']
rows_all_nan = df_b22[value_cols].isna().all(axis=1)
print(f"Number of rows missing CMP values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b22_cleaned = df_b22[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

### Volatile Organic Compounds and Trihalomethanes/MTBE

In [ ]:
file_path = '2017-2020/blood/24.P_VOCWB.xpt'

df_b24, meta = pyreadstat.read_xport(file_path)
df_b24 = standardize_id_column(df_b24)

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_VOCWB.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [ ]:
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df_b24.columns
}

df_b24 = df_b24.rename(columns=filtered_rename_dict)

In [ ]:
df_b24.head()

In [ ]:
df_b24 = df_b24.drop('voc_subsample_weight_pre_pandemic', axis=1)

In [ ]:
df_b24.isnull().sum()

In [ ]:
value_cols = [col for col in df_b24.columns if col != 'participant_id']
rows_all_nan = df_b24[value_cols].isna().all(axis=1)
print(f"Number of rows missing all values: {rows_all_nan.sum()}")

In [ ]:
# Drop rows where all value columns are NaN (excluding participant_id)
df_b24_cleaned = df_b24[~rows_all_nan].copy()

print(f"Number of rows dropped: {rows_all_nan.sum()}")

In [ ]:
df_names = [var for var in globals() if isinstance(globals()[var], pd.DataFrame)]
print(df_names)

In [ ]:
blood_dfs = [
    df_b1,
    df_b2,
    df_b3_cleaned,
    df_b4,
    df_b5,
    df_b6_cleaned,
    df_b7,
    df_b8,
    df_b9,
    df_b10,
    df_b12,
    df_b13_cleaned,
    df_b14,
    df_b15,
    df_b16_cleaned,
    df_b17_cleaned,
    df_b18_cleaned,
    df_b19_cleaned,
    df_b20_cleaned,
    df_b21_cleaned,
    df_b22_cleaned,
    df_b23,
    df_b24_cleaned
]

from functools import reduce

df_blood_combined = reduce(
    lambda left, right: pd.merge(left, right, on="participant_id", how="outer"),
    blood_dfs
)

In [ ]:
df_blood_combined.to_csv("cleaned_blood_labs_combined.csv", index=False)

In [ ]:
blood_df = pd.read_csv('cleaned_blood_labs_combined.csv')

In [ ]:
blood_df.head()

To create a relational database with MySQL, the wide table was converted to long table.

In [ ]:
blood_long = blood_df.melt(id_vars=['participant_id'],
                          var_name='test_name',
                          value_name = 'test_value')

In [ ]:
blood_long['participant_id'] = blood_long['participant_id'].astype(int)

In [ ]:
blood_long.head()

In [ ]:
blood_long.to_csv('blood_long.csv',index=False)